In [1]:
### 1. Connect google drive ###

from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [4]:
### 2. Prepare data ###

!scp "/content/gdrive/MyDrive/textDetectionDataSet/data.zip" "/content/data.zip"
!unzip "/content/data.zip" -d "/content"


Archive:  /content/data.zip
   creating: /content/data/
  inflating: /content/data/I_hate_nothing_about_u.jpg  
  inflating: /content/data/I_see_a_light_in_the_darkness.jpg  
  inflating: /content/data/aloha.jpg  
  inflating: /content/data/art.jpg   
  inflating: /content/data/aspire_to_inspire.jpg  
  inflating: /content/data/be_kind.jpg  
  inflating: /content/data/be_kind_to_one_another.jpg  
  inflating: /content/data/be_my_valentine.jpg  
  inflating: /content/data/beer_beer_beer.jpg  
  inflating: /content/data/berlin.jpg  
  inflating: /content/data/black_lives_matter_no_racism_love_not_war.jpg  
  inflating: /content/data/blank_street.jpg  
  inflating: /content/data/brand_personality_&_visual_identity.jpg  
  inflating: /content/data/burn_fat.jpg  
  inflating: /content/data/can't_hide_our_pride.jpg  
  inflating: /content/data/caution_maintain_social_distancing_at_least_6ft_distance_from_other.jpg  
  inflating: /content/data/collect_moments_not_things.jpg  
  inflating: /co

In [5]:
### 3. Install dependencies ###

!apt install tesseract-ocr
!apt install libtesseract-dev

!pip install pytesseract
!pip install Pillow
!pip install easyocr
!pip install boto3

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libacl1-dev libarchive-dev libattr1 libattr1-dev libext2fs-dev libgmp-dev
  libgmpxx4ldbl libleptonica-dev nettle-dev
Suggested packages:
  doc-base gmp-doc libgmp10-doc libmpfr-dev
The following NEW packages will be installed:
  libacl1-dev libarchive-dev libattr1-dev libext2fs-dev libgmp-dev
  libgmpxx4ldbl libleptonica-dev libtesseract-dev nettle-dev
The following packages will be upgraded:
  libattr1
1 upgraded, 9 newly installed, 0 to remove and 32 not upgraded.
Need to get 5,706 kB of archives.
After this operation, 22.4 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-update

In [21]:
### 4. text detection ###

import pytesseract
from PIL import Image
from easyocr import Reader
import boto3


reader = Reader(["en"])

access_key = 'YOUR_ACCESS_KEY'
secret_access_key = 'YOUR_SECRET_KEY'

textract_client = boto3.client('textract', aws_access_key_id=access_key, aws_secret_access_key=secret_access_key, region_name='us-east-2')

# image_path = '/content/data/it_was_all_a_dream.jpg'

def read_text_tesseract(image_path):
  text = pytesseract.image_to_string(Image.open(image_path), lang='eng')
  return text

def read_text_easyocr(image_path):
  text = ''
  results = reader.readtext(Image.open(image_path))

  for result in results:
    text = text + result[1] + ' '

  text = text[:-1]
  return text

def read_text_textract(image_path):
  with open(image_path, 'rb') as im:
    response = textract_client.detect_document_text(Document={'Bytes':im.read()})

  text = ''
  for item in response['Blocks']:
    if item['BlockType'] == 'LINE':
      text = text + item['Text'] + ''

  text = text[:-1]

  return text

In [24]:
### 5. Compare Performances ###

import os

def jaccard_similarity(sentence1, sentence2):
    # Tokenize sentences into sets of words
    set1 = set(sentence1.lower().split())
    set2 = set(sentence2.lower().split())

    # Calculate Jaccard similarity
    intersection_size = len(set1.intersection(set2))
    union_size = len(set1.union(set2))

    # Avoid division by zero if both sets are empty
    similarity = intersection_size / union_size if union_size != 0 else 0.0

    return similarity

score_tesseract = 0
score_easyocr = 0
score_textract = 0
for image_path in os.listdir('/content/data'):
  image_path = os.path.join('/content/data', image_path)

  gt = image_path[:-4].replace('_', ' ').lower()

  score_tesseract += jaccard_similarity(gt, read_text_tesseract(image_path).lower().replace('\n', '').replace('!','').replace('?','').replace('.',''))
  score_easyocr += jaccard_similarity(gt, read_text_easyocr(image_path).lower().replace('\n', '').replace('!','').replace('?','').replace('.',''))
  score_textract += jaccard_similarity(gt, read_text_textract(image_path).lower().replace('\n', '').replace('!','').replace('?','').replace('.',''))

  # break

print('Score tesseract:', score_tesseract / 100)
print('Score easyocr:', score_easyocr / 100)
print('Score textract:', score_textract / 100)

Score tesseract: 0.014819954297508478
Score easyocr: 0.22038291137826735
Score textract: 0.050141474160514406
